# 90 Commissioning (flagship)

KOH first fill, wetting, residual oxygen, formation, **grid power in**, rest, first charge, H2/O2, carbonation risk, temperature. Plot every engineer-relevant signal.

**Requirements:** `IA-SYS-020`, `IA-SYS-021`, `IA-GRD-001`.

## Sequence (verification-scaled seconds; engineering analog in hours)

| $t$ | Action |
| --- | --- |
| $0$–$600\,\mathrm{s}$ | Dry vessel, residual air $\mathrm{O_2}$, fans on |
| $600$–$2400\,\mathrm{s}$ | $6\,\mathrm{M}$ KOH make-up fill into anode pores |
| $2400$–$3600\,\mathrm{s}$ | Rest / wetting / residual $\mathrm{O_2}$ |
| $3600$–$7200\,\mathrm{s}$ | First charge from grid, OER + HER, $T$ rise |

Chemistry: IA-SYS-020/021. Grid: IA-GRD-001. Same `PlantModel` as decade 40.


## Governing equations

Fill, wetting, residual oxygen, then first charge through OER with parasitic HER.
Default $6\,\mathrm{M}$ KOH. Temperatures in $\mathrm{K}$, power in $\mathrm{W}$,
current in $\mathrm{A}$, voltage in $\mathrm{V}$.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

here = Path.cwd()
for cand in (here, here.parent):
    if (cand / "src" / "ironair").is_dir() and (cand / "plant_sim").is_dir():
        sys.path.insert(0, str(cand))
        sys.path.insert(0, str(cand / "src"))
        break

from plant_sim.bootstrap import setup_path

setup_path()
from plant_sim.params import CAPACITY_FE_OH2_MAH_G, CAPACITY_MAGNETITE_MAH_G, default_params

In [ ]:
from IPython import get_ipython

_ipy = get_ipython()
if _ipy is not None:
    _ipy.run_line_magic("matplotlib", "inline")

sns.set(
    rc={
        "axes.labelsize": 12,
        "axes.titlesize": 18,
        "figure.figsize": (11, 8.5),
        "figure.dpi": 300,
        "figure.facecolor": "w",
        "figure.edgecolor": "k",
    }
)
P = default_params()
print("KOH default", P.c_KOH_mol_m3 / 1000, "M; T_ep", P.T_ep_sim_K, "K")
print("Faraday 960/320 checks", round(CAPACITY_FE_OH2_MAH_G, 2), round(CAPACITY_MAGNETITE_MAH_G, 2))

## Simulation

In [ ]:
from plant_sim.plant import PlantModel
from plant_sim.scenarios.mission import commissioning_inputs

plant = PlantModel(P)
res = plant.simulate((0, 7200), inputs=commissioning_inputs(P), y0=plant.y0(filled=False), n_eval=360, rtol=1e-4)
t, a, y = res["t"], res["alg"], res["y"]
b = plant.blocks
plot_frame = pd.DataFrame(
    {
        "t": (t / 3600),
        "P_grid_W": (a["P_grid_W"]),
        "I_A": (a["I_cell_A"]),
        "V_V": (a["V_cell_V"]),
    }
)
plot_long = plot_frame.melt(id_vars="t", var_name="quantity", value_name="value")
fig, ax = plt.subplots()
sns.lineplot(data=plot_long, x="t", y="value", hue="quantity", ax=ax, linewidth=2.0)
ax.set_xlabel("t (h)")
ax.set_ylabel("P (W) / I (A) / V (V)")
ax.set_title("Grid power, current, voltage")
ax.text(0.0, -0.22, "90_Commissioning", transform=ax.transAxes, fontsize=8, va="top")
fig.tight_layout()
fig.subplots_adjust(bottom=0.22)
plot_frame = pd.DataFrame(
    {
        "t": (t / 3600),
        "n_OH": (y[b["ely"][0]]),
        "s_wet": (y[b["ely"][0] + 6]),
        "V_m3": (y[b["ely"][0] + 5]),
    }
)
plot_long = plot_frame.melt(id_vars="t", var_name="quantity", value_name="value")
fig, ax = plt.subplots()
sns.lineplot(data=plot_long, x="t", y="value", hue="quantity", ax=ax, linewidth=2.0)
ax.set_xlabel("t (h)")
ax.set_ylabel("n_OH / wetting / volume")
ax.set_title("KOH fill and wetting")
ax.text(0.0, -0.22, "6 M KOH make-up", transform=ax.transAxes, fontsize=8, va="top")
fig.tight_layout()
fig.subplots_adjust(bottom=0.22)
plot_frame = pd.DataFrame(
    {
        "t": (t / 3600),
        "n_H2": (a["n_H2_mol"]),
        "n_O2": (a["n_O2_oer_mol"]),
        "CE": (a["CE"]),
    }
)
plot_long = plot_frame.melt(id_vars="t", var_name="quantity", value_name="value")
fig, ax = plt.subplots()
sns.lineplot(data=plot_long, x="t", y="value", hue="quantity", ax=ax, linewidth=2.0)
ax.set_xlabel("t (h)")
ax.set_ylabel("mol / CE")
ax.set_title("Gas evolution")
ax.text(0.0, -0.22, "HER + OER on first charge", transform=ax.transAxes, fontsize=8, va="top")
fig.tight_layout()
fig.subplots_adjust(bottom=0.22)
plot_frame = pd.DataFrame(
    {
        "t": (t / 3600),
        "T_fe": (y[b["fe"][0] + 3]),
        "T_ely": (y[b["ely"][0] + 4]),
        "T_ves": (y[b["th"][0] + 2]),
    }
)
plot_long = plot_frame.melt(id_vars="t", var_name="quantity", value_name="value")
fig, ax = plt.subplots()
sns.lineplot(data=plot_long, x="t", y="value", hue="quantity", ax=ax, linewidth=2.0)
ax.set_xlabel("t (h)")
ax.set_ylabel("T (K)")
ax.set_title("Temperature rise")
ax.text(0.0, -0.22, "I*eta + formation", transform=ax.transAxes, fontsize=8, va="top")
fig.tight_layout()
fig.subplots_adjust(bottom=0.22)
plot_frame = pd.DataFrame(
    {
        "t": (t / 3600),
        "SOC": (a["SOC"]),
        "n_CO3": (y[b["ely"][0] + 2]),
        "pO2_atm": (a["p_O2_Pa"] / P.P_atm_Pa),
        "s_gdl": (a["s_gdl"]),
    }
)
plot_long = plot_frame.melt(id_vars="t", var_name="quantity", value_name="value")
fig, ax = plt.subplots()
sns.lineplot(data=plot_long, x="t", y="value", hue="quantity", ax=ax, linewidth=2.0)
ax.set_xlabel("t (h)")
ax.set_ylabel("SOC / carbonate / pO2 / s")
ax.set_title("SOC, carbonation risk, pO2, GDL saturation")
ax.text(0.0, -0.22, "US CO2; pO2 window", transform=ax.transAxes, fontsize=8, va="top")
fig.tight_layout()
fig.subplots_adjust(bottom=0.22)
print("finite", np.all(np.isfinite(y)), "nfev", res["nfev"])

## Verification against patents / SSS002

In [ ]:
assert np.all(np.isfinite(y))
print("PASS finite commissioning trajectory")